# Classic Black-Box Testing Exercise

In this exercise you practice **black-box** test design: derive tests from **requirements** (inputs/outputs, rules) **without** relying on knowledge of how the code is implemented internally.

You will work through:
- **Equivalence partitioning (EP)** — group inputs into classes and pick representatives.
- **Boundary value analysis (BVA)** — stress values at/near partition edges.
- **Combinatorial interaction testing** — same Cartesian machinery twice; only **`score_levels`** comes from **EP** vs **BVA** (see combinatorial section).

Fill in the **`TODO`** sections and run the cells **from top to bottom** (`run_ep_tests` is defined in the EP code cell).

## Software under Test (SuT) — specification

Function name: **`classify(score, track)`** → returns a `str`.

**Inputs**
- `score`: integer. Valid range is **0–100 inclusive**. Any other integer is invalid.
- `track`: string. Valid values are exactly **`"TR1"`**, **`"TR2"`**, **`"TR3"`**. Anything else is invalid.

**Invalid inputs**  
If `score` or `track` is invalid, the result must be **`"INVALID"`**.

**Valid inputs — band by score**
- If **score &lt; 40** → band **`"LOW"`**.
- Else if **score &lt; 70** → band **`"MID"`**.
- Else → band **`"HIGH"`**.

**Track rule**  
If **`track == "TR3"`** and the band would be **`"HIGH"`**, the returned value must be **`"MID"`** instead.

Design tests from this specification. The next cell contains an implementation only so you can **execute** tests; treat the box as black when choosing cases.

In [11]:
def classify(score: int, track: str) -> str:
    if score < 0 or score > 100:
        return "INVALID"
    if track not in ("TR1", "TR2", "TR3"):
        return "INVALID"

    if score < 40:
        band = "LOW"
    elif score < 70:
        band = "MID"
    else:
        band = "HIGH"

    if track == "TR3" and band == "HIGH":
        band = "MID"

    return band


## Equivalence partitioning (EP)

Split the input domain into **equivalence classes** (same expected treatment per class), then choose **one or few representatives per class**.

**TODO**
- List **invalid** classes for `score` and for `track`, and **valid** classes for the `(score, track)` behavior you care about (hint: invalid inputs collapse to one outcome).
- Write **one test per class** (or minimal set you defend) as `(score, track, expected)` tuples.
- Run `run_ep_tests` below.

In [12]:
def run_ep_tests(cases: list[tuple[int, str, str]]) -> None:
    for score, track, expected in cases:
        got = classify(score, track)
        ok = got == expected
        print(f"score={score!r}, track={track!r} -> {got!r} (expected {expected!r}) {'OK' if ok else 'FAIL'}")


# One representative per equivalence class (EP from the SuT spec).
# Invalid score: below 0 / above 100 | Invalid track | Valid bands + TR3 override.
ep_cases: list[tuple[int, str, str]] = [
    # (score, track, expected)
    (-1, "TR1", "INVALID"),  # invalid score (too small)
    (101, "TR1", "INVALID"),  # invalid score (too large)
    (50, "TR0", "INVALID"),  # invalid track (valid score)
    (10, "TR1", "LOW"),  # valid score band: LOW (any valid track behaves the same)
    (50, "TR2", "MID"),  # valid score band: MID
    (85, "TR1", "HIGH"),  # valid score band: HIGH, track not TR3
    (85, "TR3", "MID"),  # valid HIGH + TR3 → result MID (not HIGH)
]

run_ep_tests(ep_cases)


score=-1, track='TR1' -> 'INVALID' (expected 'INVALID') OK
score=101, track='TR1' -> 'INVALID' (expected 'INVALID') OK
score=50, track='TR0' -> 'INVALID' (expected 'INVALID') OK
score=10, track='TR1' -> 'LOW' (expected 'LOW') OK
score=50, track='TR2' -> 'MID' (expected 'MID') OK
score=85, track='TR1' -> 'HIGH' (expected 'HIGH') OK
score=85, track='TR3' -> 'MID' (expected 'MID') OK


## Boundary value analysis (BVA)

Place tests at **edges** of partitions: just inside/outside ranges, and neighbors of thresholds (**39/40/41**, **69/70/71**, **0/-1**, **100/101**).

**TODO**
- Add boundary-focused cases for **`score`** using a **fixed valid** `track` (e.g. `"TR1"`).
- Add at least one boundary for **`track`** invalid vs valid (e.g. string adjacent to a valid token).
- Include cases that check the **TR3 + HIGH → MID** rule near **score 70 and 100**.

In [13]:
# Boundary values: score edges with TR1; invalid track; TR3 where HIGH is demoted to MID.
bva_cases: list[tuple[int, str, str]] = [
    # --- score domain & band thresholds (track fixed to TR1) ---
    (-1, "TR1", "INVALID"),
    (0, "TR1", "LOW"),
    (39, "TR1", "LOW"),
    (40, "TR1", "MID"),
    (41, "TR1", "MID"),
    (69, "TR1", "MID"),
    (70, "TR1", "HIGH"),
    (71, "TR1", "HIGH"),
    (100, "TR1", "HIGH"),
    (101, "TR1", "INVALID"),
    # --- invalid track (valid score) ---
    (50, "", "INVALID"),
    (50, "TR4", "INVALID"),
    # --- TR3 + HIGH band → MID (edges of HIGH interval) ---
    (70, "TR3", "MID"),
    (71, "TR3", "MID"),
    (100, "TR3", "MID"),
    # --- contrast: same scores with TR2 stay HIGH ---
    (70, "TR2", "HIGH"),
    (100, "TR2", "HIGH"),
]

run_ep_tests(bva_cases)


score=-1, track='TR1' -> 'INVALID' (expected 'INVALID') OK
score=0, track='TR1' -> 'LOW' (expected 'LOW') OK
score=39, track='TR1' -> 'LOW' (expected 'LOW') OK
score=40, track='TR1' -> 'MID' (expected 'MID') OK
score=41, track='TR1' -> 'MID' (expected 'MID') OK
score=69, track='TR1' -> 'MID' (expected 'MID') OK
score=70, track='TR1' -> 'HIGH' (expected 'HIGH') OK
score=71, track='TR1' -> 'HIGH' (expected 'HIGH') OK
score=100, track='TR1' -> 'HIGH' (expected 'HIGH') OK
score=101, track='TR1' -> 'INVALID' (expected 'INVALID') OK
score=50, track='' -> 'INVALID' (expected 'INVALID') OK
score=50, track='TR4' -> 'INVALID' (expected 'INVALID') OK
score=70, track='TR3' -> 'MID' (expected 'MID') OK
score=71, track='TR3' -> 'MID' (expected 'MID') OK
score=100, track='TR3' -> 'MID' (expected 'MID') OK
score=70, track='TR2' -> 'HIGH' (expected 'HIGH') OK
score=100, track='TR2' -> 'HIGH' (expected 'HIGH') OK


## Combinatorial interaction testing (CIT)

**CIT** combines factor levels (here: **score** × **track**)—often as a full grid over those axes—and runs the SuT on each combination.

The exercises below use the **same machinery** (`run_cit`): only how you choose **`score_levels`** differs—first from **equivalence partitioning (EP)**, then from **boundary value analysis (BVA)**. The **`tracks`** axis stays `["TR1", "TR2", "TR3"]` throughout.

For many factors, **pairwise / t-wise** covering arrays reduce suite size; here the goal is to compare **EP vs BVA** level choices on one axis.

### CIT with EP — exercise

Pick **one representative `score` per equivalence band** (`LOW` / `MID` / `HIGH`): interior points **inside** each partition (not on thresholds).

That list is the **score factor’s levels**. Combine it with every `track` using **nested loops** (score × track), then **`run_cit`** prints the grid size and each **`classify`** result.

**Grid size:** 3 score levels × 3 tracks = **9** tests.

In [14]:
tracks = ["TR1", "TR2", "TR3"]

# Score axis from EP: one interior value per band (LOW / MID / HIGH).
score_levels_ep = [10, 55, 85]


def run_cit(score_levels: list[int], heading: str) -> None:
    """Outer loop: score axis. Inner loop: track axis → full grid (same as Cartesian product)."""
    print("=" * 60)
    print(heading)
    print("=" * 60)
    print("score_levels:", score_levels)
    print("Grid size:", len(score_levels) * len(tracks))
    print("--- run ---")
    for score in score_levels:
        for track in tracks:
            print((score, track), "->", classify(score, track))
    print()


run_cit(score_levels_ep, "CIT — score axis from EP (one representative per band)")


CIT — score axis from EP (one representative per band)
score_levels: [10, 55, 85]
Grid size: 9
--- run ---
(10, 'TR1') -> LOW
(10, 'TR2') -> LOW
(10, 'TR3') -> LOW
(55, 'TR1') -> MID
(55, 'TR2') -> MID
(55, 'TR3') -> MID
(85, 'TR1') -> HIGH
(85, 'TR2') -> HIGH
(85, 'TR3') -> MID



### CIT with BVA — exercise

Use **boundary-focused scores** on the **valid** domain `[0, 100]`: values on or next to **40** and **70** thresholds (e.g. `0`, `39`, `40`, `69`, `70`, `100`). These become the **score** factor’s levels; **tracks** are unchanged.

Re-run the **same** `run_cit` pipeline. **Grid size:** 6 × 3 = **18** tests—more cases than EP because each boundary is its own level.

**Requires:** run the **CIT with EP** Python cell above first so `run_cit` and `tracks` exist.

In [15]:
# Score axis from BVA (valid scores only). Requires prior EP CIT cell for `run_cit`, `tracks`.
score_levels_bva = [-1, 0, 39, 40, 69, 70, 100, 101]

run_cit(score_levels_bva, "CIT — score axis from BVA (boundary scores per band)")

CIT — score axis from BVA (boundary scores per band)
score_levels: [-1, 0, 39, 40, 69, 70, 100, 101]
Grid size: 24
--- run ---
(-1, 'TR1') -> INVALID
(-1, 'TR2') -> INVALID
(-1, 'TR3') -> INVALID
(0, 'TR1') -> LOW
(0, 'TR2') -> LOW
(0, 'TR3') -> LOW
(39, 'TR1') -> LOW
(39, 'TR2') -> LOW
(39, 'TR3') -> LOW
(40, 'TR1') -> MID
(40, 'TR2') -> MID
(40, 'TR3') -> MID
(69, 'TR1') -> MID
(69, 'TR2') -> MID
(69, 'TR3') -> MID
(70, 'TR1') -> HIGH
(70, 'TR2') -> HIGH
(70, 'TR3') -> MID
(100, 'TR1') -> HIGH
(100, 'TR2') -> HIGH
(100, 'TR3') -> MID
(101, 'TR1') -> INVALID
(101, 'TR2') -> INVALID
(101, 'TR3') -> INVALID

